In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import torch
from segmentation_models_pytorch import Unet
import kornia.augmentation as K
from torchmetrics.segmentation import MeanIoU, DiceScore
import lightning as L
from torchinfo import summary

from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from datamodules.datamodule_deadtrees import deadtrees_dm
from segmentators import RegularTrainingSegmentator
from segmentation_models_pytorch.losses import DiceLoss

VERSION = 2
CHECKPOINT_PATH = (
    f"checkpoints/deadtrees_segmentation/version_{VERSION}/checkpoints/last.ckpt"
)
PROJECT_NAME = "deadtrees_segmentation"

In [3]:
model = Unet("resnet34", in_channels=3, classes=2)
augmentation = dict(
    geom_augmentation=K.AugmentationSequential(
        K.RandomHorizontalFlip(p=0.5),
        K.RandomVerticalFlip(p=0.5),
        data_keys=["input", "mask"],
    ),
    intensity_augmentation=K.AugmentationSequential(
        K.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8),
        K.RandomGaussianNoise(mean=0.0, std=0.05, p=0.3),
        data_keys=["input"],  # solo imagen
    ),
)


segmentator = RegularTrainingSegmentator(
    model=model,
    criterion=DiceLoss(mode="binary"),
    augmentation=None,
    optimizer_cls=torch.optim.AdamW,
    optimizer_kwargs=dict(lr=1e-4, weight_decay=1e-4),
    metric=DiceScore(
        num_classes=2,
        include_background=False,
    ),
    add_channel_dim=True,
)


In [4]:
summary(segmentator, input_size=(1, 3, 256, 256), device="cpu")

Layer (type:depth-idx)                             Output Shape              Param #
RegularTrainingSegmentator                         [1, 2, 256, 256]          --
├─Unet: 1-1                                        [1, 2, 256, 256]          --
│    └─ResNetEncoder: 2-1                          [1, 3, 256, 256]          --
│    │    └─Conv2d: 3-1                            [1, 64, 128, 128]         9,408
│    │    └─BatchNorm2d: 3-2                       [1, 64, 128, 128]         128
│    │    └─ReLU: 3-3                              [1, 64, 128, 128]         --
│    │    └─MaxPool2d: 3-4                         [1, 64, 64, 64]           --
│    │    └─Sequential: 3-5                        [1, 64, 64, 64]           221,952
│    │    └─Sequential: 3-6                        [1, 128, 32, 32]          1,116,416
│    │    └─Sequential: 3-7                        [1, 256, 16, 16]          6,822,400
│    │    └─Sequential: 3-8                        [1, 512, 8, 8]            13,114,368
│   

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f"checkpoints/deadtrees_segmentation/version_{VERSION}/checkpoints",
    filename="{epoch:02d}-{step}-{val_dice:.4f}-{val_loss:.4f}",
    save_top_k=1,  # keep best 3 models
    monitor="val_dice",  # or "val_loss"
    mode="max",  # IoU → maximize
    save_last=True,  # VERY IMPORTANT for resume
)
logger = TensorBoardLogger(save_dir="logs", version=VERSION, name="minimal_example")
trainer = L.Trainer(
    max_epochs=30,
    logger=logger,
    callbacks=[checkpoint_callback],
    accelerator="gpu",
    devices=1,
    enable_progress_bar=True,
    deterministic=False,  ## Not possible for CE
    check_val_every_n_epoch=1,
    fast_dev_run=False,  # Para probar que el código corre sin errores, luego quitarlo para entrenar en serio
    overfit_batches=0,
    log_every_n_steps=1,
    precision="16-mixed",
    accumulate_grad_batches=4,
)

trainer.fit(segmentator, deadtrees_dm, ckpt_path=CHECKPOINT_PATH)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/datacuber/Documents/semantic_segmentation/src/checkpoints/deadtrees_segmentation/version_2/checkpoints exists and is not empty.
Restoring states from the checkpoint path at checkpoints/deadtrees_segmentation/version_2/checkpoints/last.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summa

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ Unet      │ 24.4 M │ train │     0 │
│ 1 │ criterion │ DiceLoss  │      0 │ train │     0 │
│ 2 │ metric    │ DiceScore │      0 │ train │     0 │
└───┴───────────┴───────────┴────────┴───────┴───────┘

Trainable params: 24.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.4 M                                                                                               
Total estimated model params size (MB): 97                                                                         
Modules in train mode: 190                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restored all states from the checkpoint at checkpoints/deadtrees_segmentation/version_2/checkpoints/last.ckpt


Output()

/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pyt
ree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=30` reached.


In [24]:
import yaml

with open("config_regular_unet.yaml", "r") as f:
    config = yaml.safe_load(f)

config

{'version': 2,
 'project_name': 'deadtrees_segmentation',
 'resume': False,
 'fast_dev_run': False,
 'overfit_batches': 1,
 'data': {'train_image_glob': '../DeadTrees/train/images/*.tif',
  'train_mask_glob': '../DeadTrees/train/masks/*.tif',
  'val_image_glob': '../DeadTrees/val/images/*.tif',
  'val_mask_glob': '../DeadTrees/val/masks/*.tif',
  'dataset_cls': 'DeadTrees',
  'white_threshold': 0.1},
 'model': {'encoder_name': 'resnet34',
  'encoder_weights': 'imagenet',
  'in_channels': 3,
  'classes': 2},
 'augmentation': {'apply': False,
  'horizontal_flip': 0.5,
  'vertical_flip': 0.5,
  'color_jitter_brightness': 0.2,
  'color_jitter_contrast': 0.2,
  'color_jitter_saturation': 0.2,
  'color_jitter_hue': 0.1,
  'collor_jitter_prob': 0.5,
  'gaussian_noise_prob': 0.5,
  'gaussian_noise_mean': 0.0,
  'gaussian_noise_std': 0.05},
 'training': {'criterion': 'DiceLoss',
  'criterion_kwargs': [{'mode': 'binary'}],
  'accelerator': 'gpu',
  'devices': 1,
  'batch_size': 16,
  'num_worker

In [19]:
VERSION = config["version"]
PROJECT_NAME = config["project_name"]

CHECKPOINT_PATH = (
    f"checkpoints/{PROJECT_NAME}/version_{VERSION}/checkpoints/last.ckpt"
    if config.get("resume")
    else None
)
print(CHECKPOINT_PATH)

None


In [20]:
from data_classes import DeadTrees, RegularDataModule

ModuleNotFoundError: No module named 'data_classes'

In [ ]:
from datamodules.data_classes import DeadTrees, RegularDataModule

In [ ]:
import datamodules.data_classes as DM


train_deadtrees_image_glob = "../DeadTrees/train/images/*.tif"
train_deadtrees_mask_glob = "../DeadTrees/train/masks/*.tif"
val_deadtrees_image_glob = "../DeadTrees/val/images/*.tif"
val_deadtrees_mask_glob = "../DeadTrees/val/masks/*.tif"
bla = getattr(DM, config["data"]["dataset_cls"])(
    train_deadtrees_image_glob, train_deadtrees_mask_glob, white_threshold=0.1
)
bla